Team Name   : **Cultuurstelsel**  
Latest Score: **3032.5**

## Bayesian reasoning for noisy agent evaluations

An agent wins 8 of 10 games after a change. The result looks promising, but ten games can also be a friendly set of seeds.

This notebook is about the gap between a good-looking score and evidence that is easier to trust. The examples focus on four questions that often appear in stochastic evaluation.

- Is the newer version better by an amount that matters?
- How much uncertainty is still present?
- Is the improvement broad or concentrated in a few opponent families?
- Where is more simulation most informative?

Most of the notebook uses synthetic data so every calculation is visible and reproducible. A later section shows aggregate results from one of my own paired experiments using the neutral names **Version 1** and **Version 2**.

I am a statistics student, and this notebook grew out of trying to reason more carefully about my own noisy agent experiments. The methods themselves are standard statistical tools. The interesting part here is how they fit together around one evaluation problem.

The main ideas are simple. Comparisons are paired on the same family, seed, and seat. Win, draw, and loss are treated as three outcomes. Shared seed noise is kept together in the bootstrap. Small opponent families are shrunk toward the broader panel with an Empirical Bayes approximation.

A local evaluation panel is still a local evaluation panel. None of these calculations turns it into a guaranteed live score, and no statistical method can repair a simulator that does not represent the question being studied.

In [ ]:
from __future__ import annotations

import math
from collections.abc import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

RNG_SEED = 20260812
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 1. The experiment comes first

A statistical model cannot repair an unfair comparison.

For candidate $c$, opponent family $f$, random seed $s$, and seat $z$, consider the point score

$$
Y_{cfsz} \in \{0, 0.5, 1\}.
$$

Two versions are most informative when they face the same $(f,s,z)$ cells. When the shared seed also aligns the relevant random inputs across versions, this is intended to act as a common random numbers design. The same favorable or difficult stochastic conditions are then experienced by both sides of the comparison.

The paired difference is

$$
D_{fsz} = Y_{Afsz} - Y_{Bfsz}.
$$

Pairing can reduce noise in the difference when the paired outcomes share useful positive dependence. The gain is a property of the coupling, not of the seed label by itself.

### The match log

One row represents one completed game from the agent's point of view.

| column | meaning |
|---|---|
| `agent` | fixed version or source hash |
| `family` | opponent behavior family |
| `seed` | random seed shared across compared versions |
| `seat` | agent seat or role |
| `score` | 1 for win, 0.5 for draw, 0 for loss |
| `margin` | optional continuous performance margin |
| `status` | `DONE`, `ERROR`, or `TIMEOUT` |
| `latency_ms` | runtime or latency measurement |
| `data_time` | date of the opponent data or replay source |

Failures belong to the result rather than outside it. A timeout can matter even when the successful games look strong.

In [ ]:
def make_dummy_panel(seed: int = RNG_SEED) -> pd.DataFrame:
    """Create a small correlated panel for a baseline and a challenger."""
    rng = np.random.default_rng(seed)
    families = [f"family_{index:02d}" for index in range(1, 13)]
    family_difficulty = dict(
        zip(families, rng.normal(0.0, 0.55, len(families)), strict=True)
    )
    challenger_interaction = {
        family: rng.normal(0.0, 0.16) for family in families
    }
    challenger_interaction["family_11"] = -0.70

    rows = []
    for random_seed in range(10_000, 10_040):
        seed_shock = rng.normal(0.0, 0.45)
        for family in families:
            for seat in (0, 1):
                seat_effect = 0.14 if seat == 1 else -0.14
                shared_match_noise = rng.normal(0.0, 0.20)

                for agent, uplift in (("baseline", 0.00), ("challenger", 0.52)):
                    interaction = (
                        challenger_interaction[family]
                        if agent == "challenger"
                        else 0.0
                    )
                    latent = (
                        -0.08
                        - family_difficulty[family]
                        + seat_effect
                        + seed_shock
                        + shared_match_noise
                        + uplift
                        + interaction
                        + rng.normal(0.0, 0.10)
                    )

                    if latent > 0.22:
                        score = 1.0
                    elif latent < -0.22:
                        score = 0.0
                    else:
                        score = 0.5

                    margin = 8_000 * (latent + rng.normal(0.0, 0.25))
                    latency = max(0.05, rng.lognormal(1.5, 0.35))
                    rows.append(
                        {
                            "agent": agent,
                            "family": family,
                            "seed": random_seed,
                            "seat": seat,
                            "score": score,
                            "margin": float(margin),
                            "status": "DONE",
                            "latency_ms": float(latency),
                            "data_time": "2026-08-01",
                        }
                    )

    return pd.DataFrame(rows)


games = make_dummy_panel()
print(games.shape)
games.head()


## 2. Before the score, the panel

The first useful output is not a posterior. It is a design audit.

Duplicated cells, missing seats, invalid scores, failed runs, and incomplete coverage can change the meaning of every statistic that follows. A clean evaluation matrix makes the later comparison much easier to interpret.

In [ ]:
def audit_design(frame: pd.DataFrame) -> dict[str, object]:
    required = {
        "agent", "family", "seed", "seat", "score",
        "margin", "status", "latency_ms", "data_time",
    }
    missing_columns = sorted(required - set(frame.columns))
    if missing_columns:
        return {"missing_columns": missing_columns}

    key = ["agent", "family", "seed", "seat"]
    duplicates = int(frame.duplicated(key).sum())
    allowed_scores = frame["score"].isin([0.0, 0.5, 1.0])
    agent_count = frame["agent"].nunique()
    coverage = frame.groupby(["family", "seed", "seat"])["agent"].nunique()

    return {
        "missing_columns": [],
        "duplicate_cells": duplicates,
        "invalid_scores": int((~allowed_scores).sum()),
        "agents": agent_count,
        "behavioral_families": frame["family"].nunique(),
        "unique_seeds": frame["seed"].nunique(),
        "seats": sorted(frame["seat"].unique().tolist()),
        "incomplete_shared_cells": int((coverage != agent_count).sum()),
        "runtime_failures": int((frame["status"] != "DONE").sum()),
        "games_per_agent": frame.groupby("agent").size().to_dict(),
    }


audit_design(games)


## 3. What the raw numbers say

Bayesian analysis should not hide the observed result.

Overall score, seat performance, weak opponent families, failures, and latency describe different parts of the same experiment. A single average can look healthy while one family remains consistently difficult.

For the lower tail, the example uses the mean score of the worst 10 percent of families. I call it a lower tail mean rather than CVaR because the quantity here is a score rather than a loss variable.

In [ ]:
def point_summary(frame: pd.DataFrame, agent: str) -> dict[str, object]:
    sample = frame.loc[frame["agent"] == agent].copy()
    family_scores = sample.groupby("family")["score"].mean().sort_values()
    tail_count = max(1, math.ceil(0.10 * len(family_scores)))
    seat_scores = sample.groupby("seat")["score"].mean()

    return {
        "games": len(sample),
        "score": sample["score"].mean(),
        "mean_margin": sample["margin"].mean(),
        "weakest_family_score": family_scores.iloc[0],
        "lower_tail_family_mean_10pct": family_scores.iloc[:tail_count].mean(),
        "seat_scores": seat_scores.to_dict(),
        "seat_gap": seat_scores.max() - seat_scores.min(),
        "runtime_failures": int((sample["status"] != "DONE").sum()),
        "latency_p99_ms": sample["latency_ms"].quantile(0.99),
    }


pd.DataFrame(
    {name: point_summary(games, name) for name in ["baseline", "challenger"]}
).T


## 4. Why 8 wins out of 10 is not the whole story

A Beta posterior is exact for a binary Bernoulli outcome, but this evaluator also allows draws.

A more direct model treats win, draw, and loss as three categories. Let

$$
(\theta_W, \theta_D, \theta_L)
$$

represent the probabilities of those outcomes. With a Dirichlet prior, the posterior remains Dirichlet after observing the counts.

The expected point score is

$$
p_{score} = \theta_W + 0.5\theta_D.
$$

This produces a posterior for the point score without pretending that a draw is literally half of a Bernoulli observation. The symmetric prior $(0.5, 0.5, 0.5)$ used below is a modeling choice. With very small samples, it is worth checking whether another reasonable prior changes the conclusion.

In [ ]:
def outcome_counts(scores: Iterable[float]) -> np.ndarray:
    values = np.asarray(list(scores), dtype=float)
    if not np.isin(values, [0.0, 0.5, 1.0]).all():
        raise ValueError("Scores must be 0, 0.5, or 1")
    return np.array(
        [
            np.sum(values == 1.0),
            np.sum(values == 0.5),
            np.sum(values == 0.0),
        ],
        dtype=float,
    )


def dirichlet_score_posterior(
    scores: Iterable[float],
    threshold: float = 0.50,
    prior: tuple[float, float, float] = (0.5, 0.5, 0.5),
    draws: int = 30_000,
    seed: int = RNG_SEED,
) -> dict[str, float]:
    counts = outcome_counts(scores)
    rng = np.random.default_rng(seed)
    posterior = rng.dirichlet(counts + np.asarray(prior), size=draws)
    point_score = posterior[:, 0] + 0.5 * posterior[:, 1]

    return {
        "wins": int(counts[0]),
        "draws": int(counts[1]),
        "losses": int(counts[2]),
        "observed_score": float((counts[0] + 0.5 * counts[1]) / counts.sum()),
        "posterior_mean_score": float(point_score.mean()),
        "credible_low_95": float(np.quantile(point_score, 0.025)),
        "credible_high_95": float(np.quantile(point_score, 0.975)),
        "probability_above_threshold": float((point_score > threshold).mean()),
    }


small_example = [1.0] * 8 + [0.0] * 2
pd.Series(dirichlet_score_posterior(small_example, threshold=0.60))


The 8 out of 10 example separates two ideas that are easy to mix together. The observed rate is 80 percent, while the uncertainty about the underlying outcome probabilities is still substantial.

The same model also gives a posterior predictive view of a future batch when the process is assumed to remain similar. This simple prediction treats future games as exchangeable and does not reproduce the full seed and family dependence structure.

In [ ]:
def posterior_predictive_batch(
    scores: Iterable[float],
    future_games: int = 50,
    draws: int = 20_000,
    seed: int = RNG_SEED,
) -> dict[str, float]:
    counts = outcome_counts(scores)
    rng = np.random.default_rng(seed)
    theta = rng.dirichlet(counts + 0.5, size=draws)

    uniforms = rng.random((draws, future_games))
    win_cut = theta[:, 0, None]
    draw_cut = (theta[:, 0] + theta[:, 1])[:, None]
    future_wins = (uniforms < win_cut).sum(axis=1)
    future_draws = ((uniforms >= win_cut) & (uniforms < draw_cut)).sum(axis=1)
    future_score = (future_wins + 0.5 * future_draws) / future_games

    return {
        "expected_future_score": float(future_score.mean()),
        "predictive_low_90": float(np.quantile(future_score, 0.05)),
        "predictive_high_90": float(np.quantile(future_score, 0.95)),
        "probability_next_batch_above_65pct": float((future_score >= 0.65).mean()),
    }


challenger_scores = games.loc[games["agent"] == "challenger", "score"]
posterior_predictive_batch(challenger_scores, future_games=50)


## 5. Comparing two versions fairly

The paired design becomes especially useful when the question changes from “is this agent good?” to “did this change help?”

When Version A and Version B are evaluated on the same family, seed, and seat, the observed difference is easier to interpret. A favorable seed is shared rather than appearing on only one side of the comparison.

In [ ]:
def paired_table(
    frame: pd.DataFrame,
    challenger: str,
    baseline: str,
) -> pd.DataFrame:
    key = ["agent", "family", "seed", "seat"]
    if frame.duplicated(key).any():
        raise ValueError("Duplicate agent, family, seed, seat cells found")

    wide = frame.pivot(
        index=["family", "seed", "seat"],
        columns="agent",
        values=["score", "margin"],
    )
    required = [
        ("score", challenger),
        ("score", baseline),
        ("margin", challenger),
        ("margin", baseline),
    ]
    if any(column not in wide.columns for column in required):
        raise ValueError("Challenger and baseline do not share the required matrix")

    paired = wide.loc[:, required].copy()
    if paired.isna().any().any():
        raise ValueError("Incomplete paired matrix. Repair coverage before inference")

    paired.columns = [
        "challenger_score",
        "baseline_score",
        "challenger_margin",
        "baseline_margin",
    ]
    paired["score_delta"] = paired["challenger_score"] - paired["baseline_score"]
    paired["margin_delta"] = paired["challenger_margin"] - paired["baseline_margin"]
    return paired.reset_index()


paired = paired_table(games, "challenger", "baseline")
paired[["score_delta", "margin_delta"]].describe()


## 6. A Bayesian view of paired uplift

The two outcomes inside a paired cell are not independent observations. One compact way to preserve the pairing is to model their full joint outcome.

Each side can win, draw, or lose, which gives nine possible paired outcome combinations. A Dirichlet prior on those nine probabilities gives posterior draws for the complete joint distribution. Each draw can then be converted into an expected paired score difference.

The resulting quantity of interest is

$$
P(\Delta > \delta \mid D),
$$

where $\delta$ is a practically meaningful improvement threshold.

This model preserves dependence within a paired cell while treating different cells as exchangeable. When many cells share the same seed, the seed-level analysis in the next section asks a different uncertainty question.

In [ ]:
def paired_dirichlet_posterior(
    paired_frame: pd.DataFrame,
    minimum_effect: float = 0.02,
    prior_per_joint_cell: float = 0.5,
    draws: int = 40_000,
    seed: int = RNG_SEED,
) -> dict[str, float]:
    levels = np.array([0.0, 0.5, 1.0])
    counts = np.zeros((3, 3), dtype=float)

    for challenger_score, baseline_score in paired_frame[
        ["challenger_score", "baseline_score"]
    ].itertuples(index=False):
        challenger_match = np.where(levels == challenger_score)[0]
        baseline_match = np.where(levels == baseline_score)[0]
        if len(challenger_match) != 1 or len(baseline_match) != 1:
            raise ValueError("Scores must be 0, 0.5, or 1")
        counts[challenger_match[0], baseline_match[0]] += 1

    rng = np.random.default_rng(seed)
    posterior = rng.dirichlet(
        counts.reshape(-1) + prior_per_joint_cell,
        size=draws,
    )
    delta_grid = (levels[:, None] - levels[None, :]).reshape(-1)
    uplift = posterior @ delta_grid

    return {
        "observed_mean_uplift": float(paired_frame["score_delta"].mean()),
        "posterior_mean_uplift": float(uplift.mean()),
        "credible_low_95": float(np.quantile(uplift, 0.025)),
        "credible_high_95": float(np.quantile(uplift, 0.975)),
        "probability_challenger_better": float((uplift > 0).mean()),
        "probability_practically_better": float((uplift > minimum_effect).mean()),
    }


paired_bayes = paired_dirichlet_posterior(paired, minimum_effect=0.02)
pd.Series(paired_bayes)


## 7. When shared seeds mean shared noise

A seed can influence many opponent families at once. In that case, the family and seat rows under the same seed are not independent replications.

For a fixed opponent panel where seeds represent the main random replication, one simple approach is to average the paired result within each seed and resample whole seeds. The resulting uncertainty describes the fixed panel under new random seeds.

A different scientific question leads to a different resampling scheme. Inference about a broader population of opponent families also needs to represent family sampling. A two-way bootstrap is not automatically “more conservative” because it changes which sources of variation are being represented.

In [ ]:
def seed_cluster_bootstrap(
    paired_frame: pd.DataFrame,
    value: str = "score_delta",
    repetitions: int = 10_000,
    seed: int = RNG_SEED,
) -> dict[str, float]:
    counts_per_seed = paired_frame.groupby("seed").size()
    if counts_per_seed.nunique() != 1:
        raise ValueError("Each seed must contain the same fixed family and seat panel")

    seed_means = paired_frame.groupby("seed")[value].mean().to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    sampled = rng.choice(
        seed_means,
        size=(repetitions, len(seed_means)),
        replace=True,
    )
    estimates = sampled.mean(axis=1)

    return {
        "point": float(seed_means.mean()),
        "one_sided_95_lcb": float(np.quantile(estimates, 0.05)),
        "two_sided_low_95": float(np.quantile(estimates, 0.025)),
        "two_sided_high_95": float(np.quantile(estimates, 0.975)),
        "bootstrap_standard_error": float(estimates.std(ddof=1)),
        "independent_seed_units": int(len(seed_means)),
    }


seed_bootstrap = seed_cluster_bootstrap(paired, "score_delta")
pd.Series(seed_bootstrap)


The paired Dirichlet model and the seed bootstrap describe uncertainty under different assumptions.

The Dirichlet model makes direct probability statements about paired outcomes. The seed bootstrap preserves dependence induced by shared random seeds. Agreement between them is informative because the same practical conclusion survives two different views of the experiment.

In [ ]:
pd.DataFrame(
    {
        "paired Bayesian model": {
            "estimate": paired_bayes["posterior_mean_uplift"],
            "low": paired_bayes["credible_low_95"],
            "high": paired_bayes["credible_high_95"],
        },
        "seed cluster bootstrap": {
            "estimate": seed_bootstrap["point"],
            "low": seed_bootstrap["two_sided_low_95"],
            "high": seed_bootstrap["two_sided_high_95"],
        },
    }
).T


## 8. Small groups and shrinkage

A family with only a few games can easily produce an extreme raw score by chance. Treating that score as equally reliable as a large family can exaggerate apparent hard counters and apparent strengths.

The example below uses an Empirical Bayes Dirichlet prior. Its center is estimated from the full panel, then each family updates that prior with its own win, draw, and loss counts.

This creates partial shrinkage toward the panel-level pattern. It is not a full hierarchical Bayesian model because the prior center and prior strength are treated as fixed rather than carrying their own posterior uncertainty.

In [ ]:
def empirical_bayes_family_scores(
    frame: pd.DataFrame,
    agent: str,
    prior_strength: float = 12.0,
    threshold: float = 0.50,
    draws: int = 20_000,
    seed: int = RNG_SEED,
) -> pd.DataFrame:
    if prior_strength <= 0:
        raise ValueError("prior_strength must be positive")

    sample = frame.loc[frame["agent"] == agent].copy()
    global_counts = outcome_counts(sample["score"])
    global_proportions = (global_counts + 0.5) / (global_counts.sum() + 1.5)
    prior_alpha = prior_strength * global_proportions

    rng = np.random.default_rng(seed)
    rows = []
    for family, family_frame in sample.groupby("family"):
        counts = outcome_counts(family_frame["score"])
        posterior = rng.dirichlet(prior_alpha + counts, size=draws)
        score_draws = posterior[:, 0] + 0.5 * posterior[:, 1]
        rows.append(
            {
                "family": family,
                "games": int(counts.sum()),
                "raw_score": float(family_frame["score"].mean()),
                "posterior_mean": float(score_draws.mean()),
                "posterior_sd": float(score_draws.std(ddof=1)),
                "credible_low_95": float(np.quantile(score_draws, 0.025)),
                "credible_high_95": float(np.quantile(score_draws, 0.975)),
                "probability_above_threshold": float((score_draws > threshold).mean()),
            }
        )

    return pd.DataFrame(rows).sort_values("posterior_mean").reset_index(drop=True)


family_posterior = empirical_bayes_family_scores(games, "challenger")
family_posterior


In [ ]:
def plot_family_shrinkage(result: pd.DataFrame) -> None:
    ordered = result.sort_values("posterior_mean").reset_index(drop=True)
    y = np.arange(len(ordered))
    lower_error = ordered["posterior_mean"] - ordered["credible_low_95"]
    upper_error = ordered["credible_high_95"] - ordered["posterior_mean"]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.errorbar(
        ordered["posterior_mean"],
        y,
        xerr=np.vstack([lower_error, upper_error]),
        fmt="o",
        capsize=3,
        label="Empirical Bayes estimate",
    )
    ax.scatter(ordered["raw_score"], y, marker="x", label="Raw score")
    ax.axvline(0.50, linewidth=1, linestyle="--")
    ax.set_yticks(y, ordered["family"])
    ax.set_xlabel("Point score")
    ax.set_title("Small family results should look uncertain")
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_family_shrinkage(family_posterior)


## 9. Where uncertainty is concentrated

Equal simulation budgets are simple, but uncertainty is rarely distributed equally across opponent families.

The diagnostic below gives more attention to families that are both uncertain and close to a decision boundary. Its score combines posterior standard deviation with uncertainty about being above that boundary.

This is a heuristic rather than Thompson sampling or an optimal computing budget allocation result. Its value is descriptive. It shows where the current posterior is least settled.

In [ ]:
def family_allocation_priority(
    frame: pd.DataFrame,
    agent: str,
    decision_boundary: float = 0.50,
    prior_strength: float = 12.0,
) -> pd.DataFrame:
    result = empirical_bayes_family_scores(
        frame,
        agent,
        prior_strength=prior_strength,
        threshold=decision_boundary,
    ).copy()

    probability_above = result["probability_above_threshold"]
    boundary_uncertainty = 1.0 - np.abs(2.0 * probability_above - 1.0)
    result["allocation_priority"] = result["posterior_sd"] * boundary_uncertainty

    total_priority = result["allocation_priority"].sum()
    if total_priority > 0:
        result["suggested_share_of_next_batch"] = (
            result["allocation_priority"] / total_priority
        )
    else:
        result["suggested_share_of_next_batch"] = 1.0 / len(result)

    return result.sort_values("allocation_priority", ascending=False).reset_index(drop=True)


family_allocation_priority(games, "challenger").head(6)


## 10. How much data is enough?

There is no universal sample size for a stochastic evaluator. The answer depends on the variability of the experiment and the precision being sought.

For a fixed opponent panel, a rough planning calculation can be made at the seed level. Let $s$ be the pilot standard deviation of the seed-level mean uplift and let $h$ be the desired normal approximation half width.

$$
N \approx \left(\frac{z s}{h}\right)^2
$$

The result is best read as a budgeting approximation rather than a final confidence procedure. As more seeds arrive, the estimated variability can change.

In [ ]:
def approximate_seeds_needed(
    paired_frame: pd.DataFrame,
    value: str = "score_delta",
    half_width: float = 0.02,
    confidence: float = 0.95,
) -> dict[str, float]:
    if half_width <= 0:
        raise ValueError("half_width must be positive")

    seed_means = paired_frame.groupby("seed")[value].mean()
    if len(seed_means) < 2:
        raise ValueError("At least two independent seeds are required")

    seed_sd = float(seed_means.std(ddof=1))
    z_value = float(norm.ppf(0.5 + confidence / 2.0))
    required = math.ceil((z_value * seed_sd / half_width) ** 2)

    return {
        "current_seeds": int(len(seed_means)),
        "pilot_seed_sd": seed_sd,
        "target_half_width": half_width,
        "approximate_required_seeds": int(required),
        "approximate_additional_seeds": int(max(0, required - len(seed_means))),
    }


approximate_seeds_needed(paired, half_width=0.02)


## 11. Repeated experiments and the winner's curse

A candidate selected from many attempts is partly selected for looking good under the particular data used during search. This is one form of the winner's curse.

A clean evaluation story separates exploratory search from final evidence. Development seeds can narrow the candidate set, fresh paired seeds can compare survivors, and a frozen version can then be evaluated on a holdout that was not part of tuning. Once a holdout influences another code change, it has started to become development information.

When many family specific hypotheses are tested formally, multiplicity becomes a separate issue. The small example below shows Holm adjustment for that setting.

In [ ]:
def holm_adjust(p_values: dict[str, float]) -> pd.DataFrame:
    ordered = sorted(p_values.items(), key=lambda item: item[1])
    total = len(ordered)
    running_max = 0.0
    rows = []

    for rank, (name, p_value) in enumerate(ordered, start=1):
        adjusted = min(1.0, (total - rank + 1) * p_value)
        running_max = max(running_max, adjusted)
        rows.append(
            {
                "hypothesis": name,
                "raw_p": p_value,
                "holm_p": running_max,
            }
        )

    return pd.DataFrame(rows).sort_values("hypothesis").reset_index(drop=True)


holm_adjust(
    {
        "family_A": 0.004,
        "family_B": 0.020,
        "family_C": 0.041,
        "family_D": 0.40,
    }
)


## 12. From evidence to a decision

Statistical evidence and a deployment decision are related, but they are not the same object. The cost of a bad promotion and the minimum improvement that matters are project choices rather than universal constants.

The example below combines three pieces of information. It looks at the posterior probability of a practically meaningful uplift, the seed-bootstrap lower bound, and runtime failures. The numerical thresholds are illustrative, so the interesting part is the structure of the decision rather than the particular cutoff.

In [ ]:
def promotion_report(
    frame: pd.DataFrame,
    challenger: str,
    baseline: str,
    minimum_effect: float = 0.02,
    required_probability: float = 0.95,
) -> dict[str, object]:
    paired_frame = paired_table(frame, challenger, baseline)
    bayes_result = paired_dirichlet_posterior(
        paired_frame,
        minimum_effect=minimum_effect,
    )
    bootstrap_result = seed_cluster_bootstrap(paired_frame, "score_delta")
    failures = int(
        (
            frame.loc[frame["agent"] == challenger, "status"]
            != "DONE"
        ).sum()
    )

    checks = {
        "posterior_practical_gain": (
            bayes_result["probability_practically_better"] >= required_probability
        ),
        "seed_bootstrap_lcb_positive": bootstrap_result["one_sided_95_lcb"] > 0,
        "no_runtime_failures": failures == 0,
    }

    return {
        "decision": "PROMOTE" if all(checks.values()) else "KEEP TESTING",
        "minimum_effect": minimum_effect,
        "required_probability": required_probability,
        "probability_practically_better": bayes_result[
            "probability_practically_better"
        ],
        "seed_bootstrap_one_sided_95_lcb": bootstrap_result[
            "one_sided_95_lcb"
        ],
        "runtime_failures": failures,
        "checks": checks,
    }


promotion_report(games, "challenger", "baseline")


## 13. A real paired experiment

The synthetic example makes every calculation reproducible. This section shows the same reasoning on aggregate results from one of my own experiments.

An older agent, **Version 1**, and a newer **Version 2** were compared on the same evaluation cells.

- 30 opponent families
- 4 shared seeds
- both seats
- 240 paired family, seed, and seat cells

The names are intentionally generic. The point is not the identity of the agent, but what a paired result looks like when the difference comes from a real experiment rather than a simulation written for the notebook.

In [ ]:
REAL_EXPERIMENT = {
    "paired_cells": 240,
    "families": 30,
    "shared_seeds": 4,
    "version_1_score": 0.4458,
    "version_2_score": 0.8917,
    "score_uplift": 0.4459,
    "improved_cells": 110,
    "unchanged_cells": 127,
    "regressed_cells": 3,
    "mean_margin_uplift": 4564.7,
    "families_improved": 24,
    "families_unchanged": 5,
    "families_regressed": 1,
    "version_2_worst_family": 0.50,
    "version_2_seat_0": 0.8917,
    "version_2_seat_1": 0.8917,
}

assert (
    REAL_EXPERIMENT["improved_cells"]
    + REAL_EXPERIMENT["unchanged_cells"]
    + REAL_EXPERIMENT["regressed_cells"]
    == REAL_EXPERIMENT["paired_cells"]
)
assert (
    REAL_EXPERIMENT["families_improved"]
    + REAL_EXPERIMENT["families_unchanged"]
    + REAL_EXPERIMENT["families_regressed"]
    == REAL_EXPERIMENT["families"]
)

real_summary = pd.DataFrame(
    [
        {
            "metric": "Point score",
            "Version 1": REAL_EXPERIMENT["version_1_score"],
            "Version 2": REAL_EXPERIMENT["version_2_score"],
            "change": REAL_EXPERIMENT["score_uplift"],
        },
        {
            "metric": "Seat 0 score",
            "Version 1": np.nan,
            "Version 2": REAL_EXPERIMENT["version_2_seat_0"],
            "change": np.nan,
        },
        {
            "metric": "Seat 1 score",
            "Version 1": np.nan,
            "Version 2": REAL_EXPERIMENT["version_2_seat_1"],
            "change": np.nan,
        },
        {
            "metric": "Worst family score",
            "Version 1": np.nan,
            "Version 2": REAL_EXPERIMENT["version_2_worst_family"],
            "change": np.nan,
        },
    ]
)

real_summary.style.format(
    {
        "Version 1": lambda x: "" if pd.isna(x) else f"{100*x:.2f}%",
        "Version 2": lambda x: "" if pd.isna(x) else f"{100*x:.2f}%",
        "change": lambda x: "" if pd.isna(x) else f"{100*x:+.2f} pp",
    }
)


### The first signal

Version 1 scored 44.58 percent and Version 2 scored 89.17 percent on the same panel. The observed paired uplift was **44.59 percentage points**. Mean margin increased by **4,564.7** in the evaluator's native units.

The average change is large, but the paired structure tells a richer story. The next question is whether the gain appeared broadly across the panel or was driven by a narrow set of matchups.

In [ ]:
def plot_real_score_comparison(result: dict[str, float]) -> None:
    labels = ["Version 1", "Version 2"]
    values = [result["version_1_score"], result["version_2_score"]]

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    bars = ax.bar(labels, values, width=0.58)
    ax.axhline(0.50, linewidth=1, linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Point score")
    ax.set_title("Point score on the same evaluation panel")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{100*x:.0f}%")

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.025,
            f"{100*value:.2f}%",
            ha="center",
            va="bottom",
        )

    ax.text(
        0.5,
        0.08,
        f"paired uplift  {100*result['score_uplift']:+.2f} percentage points",
        transform=ax.transAxes,
        ha="center",
    )
    plt.tight_layout()
    plt.show()


plot_real_score_comparison(REAL_EXPERIMENT)


### What changed inside the paired cells

The two versions were evaluated on the same 240 cells. A cell is **improved** when Version 2 produced a better discrete outcome, **regressed** when it produced a worse outcome, and **unchanged** when the outcome category stayed the same.

There were 110 improved cells, 3 regressed cells, and 127 unchanged cells. Among the 113 cells whose outcome changed, 97.35 percent moved in favor of Version 2.

In [ ]:
def plot_real_paired_changes(result: dict[str, float]) -> None:
    labels = ["Improved", "Unchanged", "Regressed"]
    counts = [
        result["improved_cells"],
        result["unchanged_cells"],
        result["regressed_cells"],
    ]

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    bars = ax.barh(labels, counts)
    ax.set_xlabel("Paired cells")
    ax.set_title("What changed across the 240 paired cells")

    for bar, count in zip(bars, counts):
        ax.text(
            count + 2,
            bar.get_y() + bar.get_height() / 2,
            str(count),
            va="center",
        )

    ax.set_xlim(0, max(counts) * 1.15)
    plt.tight_layout()
    plt.show()


discordant = REAL_EXPERIMENT["improved_cells"] + REAL_EXPERIMENT["regressed_cells"]
share_favoring_v2 = REAL_EXPERIMENT["improved_cells"] / discordant
print(f"Changed cells favoring Version 2: {100*share_favoring_v2:.2f}%")
plot_real_paired_changes(REAL_EXPERIMENT)


### A Bayesian summary of change direction

The aggregate counts still support one narrow model-based question. On this observed panel, how common are improved, unchanged, and regressed cells?

A Dirichlet model treats those three labels as categorical outcomes, mirroring the earlier win, draw, and loss example. The calculation is about **change direction**, not leaderboard rating and not the exact magnitude of score uplift.

The 240 cells are treated as exchangeable in this summary. Because the real experiment contains only four shared seeds, this is not a cluster-aware uncertainty interval. It is a model-based description of the observed change directions.

In [ ]:
def paired_change_direction_posterior(
    improved: int,
    unchanged: int,
    regressed: int,
    prior: tuple[float, float, float] = (0.5, 0.5, 0.5),
    draws: int = 80_000,
    seed: int = RNG_SEED,
) -> tuple[pd.Series, np.ndarray]:
    counts = np.array([improved, unchanged, regressed], dtype=float)
    rng = np.random.default_rng(seed)
    posterior = rng.dirichlet(counts + np.asarray(prior), size=draws)
    net_direction = posterior[:, 0] - posterior[:, 2]

    summary = pd.Series(
        {
            "posterior_mean_improved": posterior[:, 0].mean(),
            "posterior_mean_unchanged": posterior[:, 1].mean(),
            "posterior_mean_regressed": posterior[:, 2].mean(),
            "probability_improve_rate_above_regress_rate": (
                posterior[:, 0] > posterior[:, 2]
            ).mean(),
            "net_direction_low_95": np.quantile(net_direction, 0.025),
            "net_direction_high_95": np.quantile(net_direction, 0.975),
        }
    )
    return summary, posterior


real_direction_summary, real_direction_draws = paired_change_direction_posterior(
    REAL_EXPERIMENT["improved_cells"],
    REAL_EXPERIMENT["unchanged_cells"],
    REAL_EXPERIMENT["regressed_cells"],
)

real_direction_summary


Under the exchangeable cell Dirichlet model, essentially all posterior mass places the improvement frequency above the regression frequency. A more informative summary is the interval for

$$
P(\text{improved}) - P(\text{regressed}).
$$

For these aggregate counts, the 95 percent posterior interval is about 37.7 to 50.9 percentage points.

This is a different quantity from the 44.59 percentage point score uplift. The interval above describes how often cells move in a favorable rather than unfavorable direction. Score uplift also depends on the size of each discrete change.

In [ ]:
def plot_change_direction_posterior(draws: np.ndarray) -> None:
    names = ["Improved", "Unchanged", "Regressed"]
    means = draws.mean(axis=0)
    lows = np.quantile(draws, 0.025, axis=0)
    highs = np.quantile(draws, 0.975, axis=0)
    positions = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    ax.errorbar(
        means,
        positions,
        xerr=np.vstack([means - lows, highs - means]),
        fmt="o",
        capsize=4,
    )
    ax.set_yticks(positions, names)
    ax.set_xlim(0, 0.65)
    ax.set_xlabel("Posterior probability of a cell category")
    ax.set_title("Dirichlet summary of change direction")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{100*x:.0f}%")
    plt.tight_layout()
    plt.show()


plot_change_direction_posterior(real_direction_draws)


### Was the gain concentrated in a few families?

Version 2 improved in 24 of the 30 opponent families, regressed in 1, and was unchanged in 5.

That distribution matters because a large aggregate score can hide a narrow improvement. Here the directional gain appeared across most of the fixed panel. The weakest Version 2 family still scored 50 percent, so the lower tail remained less convincing than the overall average.

In [ ]:
def plot_real_family_changes(result: dict[str, float]) -> None:
    labels = ["Improved", "Unchanged", "Regressed"]
    counts = [
        result["families_improved"],
        result["families_unchanged"],
        result["families_regressed"],
    ]

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    bars = ax.bar(labels, counts)
    ax.set_ylabel("Opponent families")
    ax.set_title("Direction of change across 30 opponent families")

    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            count + 0.45,
            str(count),
            ha="center",
        )

    ax.set_ylim(0, max(counts) * 1.15)
    plt.tight_layout()
    plt.show()


plot_real_family_changes(REAL_EXPERIMENT)


### Reading the real result

Several descriptive signals point in the same direction. Version 2 gained 44.59 percentage points on the same panel, improved cells outnumbered regressions by 110 to 3, 24 of 30 families improved, and both Version 2 seats scored 89.17 percent.

In this experiment, those results were strong enough to justify moving Version 2 to live evaluation. The evidence is best described as **strong local evidence of improvement** rather than a guarantee about live rating.

Only aggregate results are shown here. That is enough for the descriptive plots and the exchangeable cell Dirichlet summary, but not for reconstructing seed-cluster uncertainty or per family posteriors. Those require the original cell-level observations.

### A later live snapshot

The controlled panel was not the end of the story. After Version 2 moved to the live Kaggriculture environment, the two versions also separated in the live score.

On **12 August 2026**, the snapshot I recorded was **2823.6 for Version 1** and **3032.5 for Version 2**. The difference was **208.9 points**.

This live gap is useful as a follow up observation, but it answers a different question from the paired experiment. The local panel compares both versions under matched conditions. A live score is produced by the environment each version actually encounters over time. The fact that both point in the same direction is encouraging without making them interchangeable pieces of evidence.

In [ ]:
LIVE_SCORE_SNAPSHOT = {
    "date": "2026-08-12",
    "Version 1": 2823.6,
    "Version 2": 3032.5,
}

live_gap = LIVE_SCORE_SNAPSHOT["Version 2"] - LIVE_SCORE_SNAPSHOT["Version 1"]

pd.Series(
    {
        "Version 1 live score": LIVE_SCORE_SNAPSHOT["Version 1"],
        "Version 2 live score": LIVE_SCORE_SNAPSHOT["Version 2"],
        "absolute gap": live_gap,
    }
)

In [ ]:
def plot_live_score_snapshot(snapshot: dict[str, float | str]) -> None:
    labels = ["Version 1", "Version 2"]
    values = [float(snapshot[label]) for label in labels]
    gap = values[1] - values[0]

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    bars = ax.bar(labels, values, width=0.58)
    ax.set_ylabel("Live Kaggriculture score")
    ax.set_title(f"Live score snapshot on {snapshot['date']}")

    ax.set_ylim(0, max(values) * 1.10)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 22,
            f"{value:.1f}",
            ha="center",
            va="bottom",
        )

    ax.text(
        0.5,
        0.08,
        f"observed gap  {gap:+.1f} points",
        transform=ax.transAxes,
        ha="center",
    )
    plt.tight_layout()
    plt.show()


plot_live_score_snapshot(LIVE_SCORE_SNAPSHOT)

### The broader lesson

The most useful part of the analysis is not any single Bayesian number.

The paired design defines a fair comparison. Raw scores show the size of the change. Cell directions show how consistently the change appears. Family and seat summaries show where the average may be hiding structure. Bayesian and resampling methods then describe uncertainty under explicit assumptions.

The order matters because uncertainty is easier to interpret after the experiment itself is clear.

## 14. The same structure with a real evaluator log

The synthetic panel and a real evaluator log can share the same basic schema. One row represents one game, and the comparison keys remain family, seed, and seat.

The short code block below shows the expected columns and an example Kaggle path. It is included to make the data structure concrete rather than to prescribe a particular workflow.

In [ ]:
# Example Kaggle path
# real_games = pd.read_csv("/kaggle/input/your-agent-evaluation/games.csv")
# print(audit_design(real_games))
# real_paired = paired_table(real_games, "challenger_hash", "baseline_hash")
# print(paired_dirichlet_posterior(real_paired, minimum_effect=0.02))
# print(seed_cluster_bootstrap(real_paired, "score_delta"))
# display(empirical_bayes_family_scores(real_games, "challenger_hash"))
# display(family_allocation_priority(real_games, "challenger_hash"))

EXPECTED_COLUMNS = [
    "agent",
    "family",
    "seed",
    "seat",
    "score",
    "margin",
    "status",
    "latency_ms",
    "data_time",
]

pd.DataFrame(columns=EXPECTED_COLUMNS)


## 15. What makes a result interpretable

A score becomes much easier to understand when the experiment around it is visible. Useful context includes the exact agent version, the evaluator schema, how seeds were generated, how opponent families were defined, the data cutoff, failures and timeouts, the practical effect size of interest, and the limitations of the local panel.

That context separates three questions that are often mixed together. What happened on the observed panel? How uncertain is the estimate under the chosen model? How far can the result reasonably generalize beyond the panel?

A local replay panel does not imply a universal live result. An Empirical Bayes shrinkage estimate is not a full hierarchical model. A posterior probability is meaningful only relative to its likelihood, prior, and experimental design.

The statistical ideas in this notebook are standard rather than novel. The main takeaway is simply that stochastic evaluation becomes easier to reason about when the comparison is paired, the observed outcome is modeled directly, dependence is kept visible, and uncertainty is shown next to the point estimate.